# Scheduled-interval flow


Load the data.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go


def find_train_file():
    here = Path.cwd().resolve()

    candidates = []
    for root in [here, *here.parents]:
        candidates.extend([
            root / "data" / "processed" / "Train_cleaned.csv",
            root / "data" / "raw" / "zindi_flight_delay" / "Train.csv",
            root / "Train_cleaned.csv",
            root / "Train.csv",
        ])

    for path in candidates:
        if path.exists():
            return path

    raise FileNotFoundError(
        "Could not find Train_cleaned.csv or Train.csv. "
        "Run the notebook from inside the project repository."
    )


TRAIN_PATH = find_train_file()
train = pd.read_csv(TRAIN_PATH)

required = {"DEPSTN", "ARRSTN", "target"}
missing = required.difference(train.columns)

if missing:
    raise KeyError(f"Missing required columns: {sorted(missing)}")

print("Train source:", TRAIN_PATH)
print("Rows:", f"{len(train):,}")


FileNotFoundError: Could not find Train_cleaned.csv or Train.csv. Run the notebook from inside the project repository.

Build the scheduled interval and filter invalid records.


In [ ]:
# Build the scheduled interval
# Use the existing interval column when available.
if "scheduled_interval_min" not in train.columns:
    if not {"STD", "STA"}.issubset(train.columns):
        raise KeyError(
            "Need either 'scheduled_interval_min' or both 'STD' and 'STA' "
            "to construct the scheduled interval."
        )

    train["STD"] = pd.to_datetime(train["STD"], errors="raise")
    # Raw STA values may use dots between H.M.S.
    try:
        train["STA"] = pd.to_datetime(train["STA"], errors="raise")
    except Exception:
        train["STA"] = pd.to_datetime(
            train["STA"],
            format="%Y-%m-%d %H.%M.%S",
            errors="raise",
        )

    raw_interval = (
        train["STA"] - train["STD"]
    ).dt.total_seconds().div(60)

    # Flag same-airport records with implausibly long intervals.
    timing_anomaly = (
        train["DEPSTN"].astype("string").eq(
            train["ARRSTN"].astype("string")
        )
        & raw_interval.ge(24 * 60)
    )

    train["timing_anomaly"] = timing_anomaly
    train["scheduled_interval_min"] = raw_interval.mask(timing_anomaly)

else:
    train["scheduled_interval_min"] = pd.to_numeric(
        train["scheduled_interval_min"],
        errors="coerce",
    )
    if "timing_anomaly" not in train.columns:
        train["timing_anomaly"] = False
    else:
        # Normalise the anomaly flag.
        train["timing_anomaly"] = (
            train["timing_anomaly"]
            .astype("string")
            .str.strip()
            .str.lower()
            .map({
                "true": True, "false": False,
                "1": True, "0": False,
                "yes": True, "no": False,
            })
            .fillna(False)
            .astype(bool)
        )

train["target"] = pd.to_numeric(train["target"], errors="coerce")

interval_data = train.loc[
    train["scheduled_interval_min"].notna()
    & train["target"].notna()
    & ~train["timing_anomaly"]
    & train["scheduled_interval_min"].gt(0)
    & train["scheduled_interval_min"].lt(24 * 60)
].copy()

if interval_data.empty:
    raise ValueError("No usable rows remain for the scheduled-interval flow.")


Create the flow categories.


In [ ]:
# Keep the main airports readable
TOP_DEP_INTERVAL_FLOW = 8
TOP_ARR_INTERVAL_FLOW = 8

top_dep_interval = (
    interval_data["DEPSTN"]
    .astype("string")
    .value_counts()
    .head(TOP_DEP_INTERVAL_FLOW)
    .index
)

top_arr_interval = (
    interval_data["ARRSTN"]
    .astype("string")
    .value_counts()
    .head(TOP_ARR_INTERVAL_FLOW)
    .index
)

flow_interval = interval_data.copy()
flow_interval["Departure"] = np.where(
    flow_interval["DEPSTN"].astype("string").isin(top_dep_interval),
    flow_interval["DEPSTN"].astype("string"),
    "Other",
)
flow_interval["Arrival"] = np.where(
    flow_interval["ARRSTN"].astype("string").isin(top_arr_interval),
    flow_interval["ARRSTN"].astype("string"),
    "Other",
)

# Scheduled interval groups
interval_order = [
    "<60 min",
    "60–119 min",
    "120–179 min",
    "180–239 min",
    "240–359 min",
    "360+ min",
]

flow_interval["Scheduled interval"] = pd.cut(
    flow_interval["scheduled_interval_min"],
    bins=[-np.inf, 60, 120, 180, 240, 360, np.inf],
    labels=interval_order,
    right=False,
    include_lowest=True,
).astype("string")

# Delay percentile groups
p50, p80, p90, p95 = [
    float(train["target"].quantile(q))
    for q in (0.50, 0.80, 0.90, 0.95)
]

delay_order = [
    f"≤P50 · ≤{p50:.0f} min",
    f"P50–P80 · >{p50:.0f} to ≤{p80:.0f} min",
    f"P80–P90 · >{p80:.0f} to ≤{p90:.0f} min",
    f"P90–P95 · >{p90:.0f} to ≤{p95:.0f} min",
    f"P95–P100 · >{p95:.0f} min",
]


t = flow_interval["target"].to_numpy()
flow_interval["Observed delay"] = np.select(
    [
        t <= p50,
        (t > p50) & (t <= p80),
        (t > p80) & (t <= p90),
        (t > p90) & (t <= p95),
        t > p95,
    ],
    delay_order,
    default=delay_order[0],
)

# Keep only groups present in the data.
present_intervals = set(flow_interval["Scheduled interval"].dropna())
interval_order = [x for x in interval_order if x in present_intervals]

present_delays = set(flow_interval["Observed delay"])
delay_order = [x for x in delay_order if x in present_delays]


Build the Sankey links.


In [ ]:
# Colours
interval_palette = {
    "<60 min": "#3568A8",
    "60–119 min": "#4DA6A6",
    "120–179 min": "#58A65C",
    "180–239 min": "#D5B72E",
    "240–359 min": "#E9802E",
    "360+ min": "#D94C4C",
}

delay_palette_full = {
    f"≤P50 · ≤{p50:.0f} min": "#2F6FB0",
    f"P50–P80 · >{p50:.0f} to ≤{p80:.0f} min": "#3FA69A",
    f"P80–P90 · >{p80:.0f} to ≤{p90:.0f} min": "#B8C738",
    f"P90–P95 · >{p90:.0f} to ≤{p95:.0f} min": "#F39A2E",
    f"P95–P100 · >{p95:.0f} min": "#C83D4D",
}
delay_palette = {k: delay_palette_full[k] for k in delay_order}

def rgba(hex_color, alpha=0.31):
    h = hex_color.lstrip("#")
    r, g, b = int(h[:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

# Stage order
dep_order = flow_interval["Departure"].value_counts().index.tolist()
arr_order = flow_interval["Arrival"].value_counts().index.tolist()

if "Other" in dep_order:
    dep_order = [x for x in dep_order if x != "Other"] + ["Other"]
if "Other" in arr_order:
    arr_order = [x for x in arr_order if x != "Other"] + ["Other"]

stage_share = {
    "Departure": flow_interval["Departure"].value_counts(normalize=True).mul(100),
    "Scheduled interval": flow_interval["Scheduled interval"].value_counts(normalize=True).mul(100),
    "Arrival": flow_interval["Arrival"].value_counts(normalize=True).mul(100),
    "Observed delay": flow_interval["Observed delay"].value_counts(normalize=True).mul(100),
}

# Nodes
node_keys, node_labels, node_colors, node_x, node_y = [], [], [], [], []

def add_nodes(stage, values, x, color_fn, top=0.06, bottom=0.94):
    if not values:
        return
    ys = np.linspace(top, bottom, len(values))
    for value, yv in zip(values, ys):
        value = str(value)
        node_keys.append((stage, value))
        node_labels.append(
            f"{value} · {stage_share[stage].get(value, 0):.1f}%"
        )
        node_colors.append(color_fn(value))
        node_x.append(float(x))
        node_y.append(float(yv))

add_nodes("Departure", dep_order, 0.03, lambda _: "#DCE8F4")
add_nodes(
    "Scheduled interval",
    interval_order,
    0.36,
    lambda x: interval_palette[x],
    0.10,
    0.90,
)
add_nodes("Arrival", arr_order, 0.69, lambda _: "#DDECDD")
add_nodes(
    "Observed delay",
    delay_order,
    0.97,
    lambda x: delay_palette[x],
    0.18,
    0.82,
)

node_index = {key: i for i, key in enumerate(node_keys)}

# Links
sources, targets, values, link_colors, custom = [], [], [], [], []

def add_transition(left_col, right_col, color_by):
    group_cols = list(dict.fromkeys([left_col, right_col, color_by]))

    grouped = (
        flow_interval
        .dropna(subset=group_cols)
        .groupby(group_cols, observed=True, dropna=False)["target"]
        .agg(n="size", median_delay="median")
        .reset_index()
    )

    for _, row in grouped.iterrows():
        left = str(row[left_col])
        right = str(row[right_col])
        colour_key = str(row[color_by])

        left_key = (left_col, left)
        right_key = (right_col, right)

        if left_key not in node_index or right_key not in node_index:
            continue

        sources.append(node_index[left_key])
        targets.append(node_index[right_key])
        values.append(int(row["n"]))

        if color_by == "Scheduled interval":
            colour = interval_palette[colour_key]
        else:
            colour = delay_palette[colour_key]

        link_colors.append(rgba(colour, 0.31))
        custom.append(
            f"<b>{left} → {right}</b><br>"
            f"Flights: {int(row['n']):,}<br>"
            f"Median delay: {row['median_delay']:.1f} min"
        )

add_transition("Departure", "Scheduled interval", "Scheduled interval")
add_transition("Scheduled interval", "Arrival", "Scheduled interval")
add_transition("Arrival", "Observed delay", "Observed delay")

if not values:
    raise ValueError("No Sankey links were created. Check the input data.")


Show the full-period flow.


In [ ]:
# Plot
fig = go.Figure(
    go.Sankey(
        arrangement="fixed",
        domain=dict(x=[0.0, 1.0], y=[0.05, 0.70]),
        textfont=dict(size=13, color="#24324A", family="Arial"),
        node=dict(
            pad=20,
            thickness=18,
            label=node_labels,
            color=node_colors,
            line=dict(color="rgba(50,60,72,0.22)", width=0.7),
            x=node_x,
            y=node_y,
            hovertemplate="%{label}<extra></extra>",
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color=link_colors,
            customdata=custom,
            hovertemplate="%{customdata}<extra></extra>",
        ),
    )
)

annotations = [
    dict(
        x=0.03, y=0.84, xref="paper", yref="paper",
        text="<b>1 · Departure airport</b>",
        showarrow=False, xanchor="left",
        font=dict(size=15, color="#1F355E"),
    ),
    dict(
        x=0.03, y=0.795, xref="paper", yref="paper",
        text="Where supported traffic originates",
        showarrow=False, xanchor="left",
        font=dict(size=10, color="#6A7280"),
    ),
    dict(
        x=0.36, y=0.84, xref="paper", yref="paper",
        text="<b>2 · Scheduled interval</b>",
        showarrow=False, xanchor="center",
        font=dict(size=15, color="#1F355E"),
    ),
    dict(
        x=0.36, y=0.795, xref="paper", yref="paper",
        text="Duration-like schedule regime",
        showarrow=False, xanchor="center",
        font=dict(size=10, color="#6A7280"),
    ),
    dict(
        x=0.69, y=0.84, xref="paper", yref="paper",
        text="<b>3 · Arrival airport</b>",
        showarrow=False, xanchor="center",
        font=dict(size=15, color="#1F355E"),
    ),
    dict(
        x=0.69, y=0.795, xref="paper", yref="paper",
        text="Where supported traffic terminates",
        showarrow=False, xanchor="center",
        font=dict(size=10, color="#6A7280"),
    ),
    dict(
        x=0.97, y=0.84, xref="paper", yref="paper",
        text="<b>4 · Observed delay</b>",
        showarrow=False, xanchor="right",
        font=dict(size=15, color="#1F355E"),
    ),
    dict(
        x=0.97, y=0.795, xref="paper", yref="paper",
        text="Global P50 / P80 / P90 / P95 tiers · EDA only",
        showarrow=False, xanchor="right",
        font=dict(size=10, color="#8A4A40"),
    ),
    dict(
        x=0.50, y=-0.095, xref="paper", yref="paper",
        text=(
            "<b>How to read:</b> width = flights · "
            "interval colours = schedule regime · "
            "final colours = observed-delay tier"
        ),
        showarrow=False, xanchor="center",
        font=dict(size=10.5, color="#555E69"),
    ),
]

fig.update_layout(
    title=dict(
        text=(
            "<b>Scheduled-interval flow</b>"
            "<br><sup>Which corridors occupy which schedule-duration regimes, "
            "and where does delay accumulate?</sup>"
        ),
        x=0.025,
        y=0.985,
        xanchor="left",
        yanchor="top",
        font=dict(size=21, color="#1F355E"),
    ),
    template="plotly_white",
    width=1780,
    height=1060,
    margin=dict(l=95, r=110, t=225, b=170),
    annotations=annotations,
)

fig.show()
